# 07 · Request types by age and Electronics cases

This notebook replicates the Request types by age and Electronics cases paragraphs of the report's Results section.

> Request types by age: Young machines run 8.0 tickets per 100 machine-months vs. 4.8 for older ones (1.66 times), with higher rates in nearly every request group. Within model, Electronics' share is 1.31 times as large on young machines (mostly Touchscreen) and Mechanical's 0.70 times. 80% of the 1,447 extra tickets come from young machines running above older ones of the same model. Touchscreen tickets cluster on 2022–23 installs (1.54 per 100, 3.9 times the 2024–26 rate).
>
> Electronics cases: Per machine-month, Electronics runs 1.71 times as often on young machines, near the 1.66 for all tickets. Touchscreen stands out at 4.2 times: of its 411 tickets on young machines, 344 are connector reseats or replacements, 402 are on the four 2.0 models (LVD-Remote-2.0, MV-2.0, LVRR-WP-2.0, and SV-48-2.0), and 315 are on 2022–23 installs. Its rate steps from 0.36 to 1.61 per 100 between Jan–Apr and May–Aug 2022 installs, which are nearly the same age, so the step points to a production, part, or installation change. The case list has all 1,605 unplanned Electronics cases on the fleet machines behind slide 4.

We also check two numbers that the report's Definitions give for terms these paragraphs use: the 15 comparison models behind "within model" and the 1,156 extra tickets behind the 80%. Three numbers need more of IPA's case export and installed-base list than Part A's tables keep (the 344 connector tickets and the step from 0.36 to 1.61), so we read them from the aggregate tables that the Electronics case list's builder writes.

One number does not hold on the base that the report's Definitions describe. `followups.py` computes Electronics' within-model share ratio, 1.31, on each model's tickets in nine of the ten request groups, every group but Not categorized. On all of a model's unplanned tickets, which is how the Definitions read, it is 1.32. We show both values and list the 1.31 in a table of its own, outside the check.

**How it works.** `followups.py` (Part A of the follow-ups) counts the unplanned tickets on IPA's fleet machines by request group, Category, model, age group, and install year, compares young machines (under 4 years) with older ones, and splits young machines' extra tickets into a within-model part, a model-mix part, and a part on the models outside the 15 comparison models. We recompute the counts, rates, ratios, and splits from its tables by age band, model, and install year, from the per-machine table of `fleet_exposure.py`, and from the per-ticket table `followups/rp_pairs.csv`. For Electronics and Mechanical, we rebuild each comparison model's tickets from these tables, using the script's within-model rate ratio to place the few tickets they do not show, and for Touchscreen we rebuild them from its counts by install year. Beyond that rate ratio, we read from the script's tables only the bootstrap intervals and permutation-test results, which need its resampling, and the within-model excesses of the Electronics Categories other than Touchscreen, which need each comparison model's tickets by Category.

**Re-run.** With `RERUN = True`, we re-run `verification/fleet_exposure.py`, which takes about 35 s, then `verification/followups.py`, which takes about 20 s, and then `viz/deliverables/build_electronics_cases.py --tables`, which takes about 10 s. They rewrite the 21 `fleet/fx_*` files, the 23 `followups/` files (the `ag_*` and `rp_*` tables and `followups_summary.json`), and the builder's three `electronics/ec_*.csv` tables in a scratch copy of `verification/outputs/`, and we then compare each of the 47 files with the committed one. `fleet_exposure.py` also writes its real-name table to a scratch copy of `data/private/`, which the notebook does not read and which is deleted when the kernel exits. With `--tables`, the builder runs its data checks against the `followups/` tables, checks that its tables hold no real identifier, and writes no workbook. Its three aggregate tables hold ticket counts (`ec_counts.csv`), each subset's tickets, machines, and accounts (`ec_subsets.csv`), and each install window's machines, young machines, machine-months, Touchscreen tickets, and rate per 100 machine-months (`ec_edge.csv`). `docs/09_followups_protocol.md` (§4 and §12) and `docs/10_followups_results.md` (items 1 to 4, §2, and §6) describe this part of the analysis.

**Used as committed, without re-running.** The scripts also read `fu/fu_ticket_last10.csv` and `fu/fu_devices.csv`, the ticket-to-device map that `db_followup.py` wrote from the telemetry database, which the notebooks do not re-read. The loader of `followups.py`, which the builder also uses, reads them to give each machine its telemetry type, which these paragraphs do not use. `fleet_exposure.py` reads `htm_device_exposure.csv`, which `high_ticket_machines.py` writes from the telemetry-database extracts (notebook 02 re-runs it), for docs/05's chronic outliers and its interim telemetry-type rates, which these paragraphs do not use either. Before any result, `followups.py` compares its run with committed outputs of `fleet_exposure.py` (re-run here first), `reproduce_sql.py`, `reproduce_cleanroom.py`, and `audit_case_logic.py`, and it stops if any of them differ. The Electronics case list itself, which the builder writes with IPA's case numbers and notes, is never committed, and this notebook does not read it.

In [1]:
RERUN = True  # False: recompute from the committed outputs without re-running the scripts
import pandas as pd
import replicate as R
OUT = R.outputs(RERUN)

RERUN is True: the scripts write into a scratch copy of verification/outputs/.


In [2]:
if RERUN:
    R.rerun(OUT, 'fleet_exposure.py')
    R.rerun(OUT, 'followups.py')
    R.rerun(OUT, 'viz/deliverables/build_electronics_cases.py', '--tables')
R.compare(OUT)

fleet_exposure.py: exit status 0, 36 s
    27/27 checks pass


followups.py: exit status 0, 20 s
    67/67 checks pass; 19 s


viz/deliverables/build_electronics_cases.py --tables: exit status 0, 7 s
    checks pass: Electronics tables written to electronics/
47 files written: 47 identical.


,file written by the re-run,compared with the committed file
0,electronics/ec_counts.csv,identical
1,electronics/ec_edge.csv,identical
2,electronics/ec_subsets.csv,identical
3,fleet/fx_age_rates.csv,identical
4,fleet/fx_country_rates.csv,identical
5,fleet/fx_coverage_rates.csv,identical
6,fleet/fx_customer_lorenz.csv,identical
7,fleet/fx_customer_rates.csv,identical
8,fleet/fx_family_rates.csv,identical
9,fleet/fx_machine_flags.csv,identical


## Young machines: 8.0 tickets per 100 machine-months vs. 4.8 for older ones, 1.66 times

Young machines are under 4 years old at the midpoint of their installed time in the period (the age bands <1, 1–2, and 2–4), and the rate is unplanned tickets per 100 installed machine-months. We add up each machine's tickets from `fleet/fx_machine_flags.csv`, which has one row per fleet machine with a ticket, and the machine-months from `followups/ag_support.csv`, which has one row per model and age group and also counts the machines without a ticket.

In [3]:
import numpy as np

summary = R.read_json(OUT, 'followups/followups_summary.json')
definitions = summary['definitions']
YOUNG_BANDS = definitions['young_bands']  # '<1', '1-2' and '2-4': under 4 years
AGES = ['Under 4', '4+']


def age_group(bands):
    """The age group of each age band."""
    return np.where(pd.Series(bands).isin(YOUNG_BANDS), AGES[0], AGES[1])


machines = R.read_csv(OUT, 'fleet/fx_machine_flags.csv')  # one row per fleet machine with an unplanned ticket
machines['age_group'] = age_group(machines.age_band)
support = R.read_csv(OUT, 'followups/ag_support.csv')  # one row per model and age group, over every fleet machine

by_age = pd.DataFrame({
    'machines': support.groupby('age_group').machines.sum(),
    'machines with a ticket': machines.groupby('age_group').size(),
    'installed machine-months': support.groupby('age_group').months.sum(),
    'unplanned tickets': machines.groupby('age_group').tickets.sum()}).reindex(AGES)
by_age['per 100 machine-months'] = 100 * by_age['unplanned tickets'] / by_age['installed machine-months']
rate_young, rate_old = by_age['per 100 machine-months']
months_young, months_old = by_age['installed machine-months']
ratio_all = rate_young / rate_old

listed = support.set_index(['model', 'age_group']).tickets
per_model = machines.groupby(['model', 'age_group']).tickets.sum()
assert per_model.reindex(listed.index, fill_value=0).eq(listed).all() and per_model.sum() == listed.sum(), (
    "the per-machine tickets do not add up to ag_support.csv's")
print(f"{int(by_age['unplanned tickets'].sum()):,} unplanned tickets on {int(by_age.machines.sum()):,} fleet machines. "
      "The per-machine tickets add up to ag_support.csv's for every model and age group.")
print(f'Young vs. older rate: {ratio_all:.4f}')
by_age.round(2)

6,750 unplanned tickets on 8,946 fleet machines. The per-machine tickets add up to ag_support.csv's for every model and age group.
Young vs. older rate: 1.6633


,machines,machines with a ticket,installed machine-months,unplanned tickets,per 100 machine-months
age_group,,,,,
Under 4,3987,1778,45103.38,3629,8.05
4+,4959,1679,64517.91,3121,4.84


## Higher rates in nearly every request group

`followups/ag_band_rates.csv` has each request group's tickets and the machine-months in each of the seven age bands, so we add up the three young bands and the four older ones and divide, all models together. The 95% intervals of the ratios come from the script's bootstrap over accounts (2,000 draws), so we read them from `followups/ag_type_rates.csv`. We read "nearly every" as every one of the ten groups having a higher rate on young machines, with the interval excluding 1 for all groups but two.

In [4]:
bands = R.read_csv(OUT, 'followups/ag_band_rates.csv')  # one row per age band and request group
bands['age_group'] = age_group(bands.age_band)
GROUPS = definitions['groups']  # the ten request groups
ALL = 'All unplanned tickets'

sums = bands.groupby(['type', 'age_group'])[['tickets', 'months']].sum()
young_sums, old_sums = sums.xs(AGES[0], level='age_group'), sums.xs(AGES[1], level='age_group')
crude = pd.DataFrame({'young tickets': young_sums.tickets, 'older tickets': old_sums.tickets,
                      'young per 100': 100 * young_sums.tickets / young_sums.months,
                      'older per 100': 100 * old_sums.tickets / old_sums.months}).reindex(GROUPS + [ALL])
crude['ratio'] = crude['young per 100'] / crude['older per 100']

rates = R.read_csv(OUT, 'followups/ag_type_rates.csv')  # crude rates by request type, with the bootstrap intervals
boot = rates[rates.level.isin(['group', 'all'])].set_index('type').loc[crude.index]
crude['95% interval'] = [R.span(lo, hi, 2) for lo, hi in boot[['rate_ratio_lo', 'rate_ratio_hi']].to_numpy()]

lower = boot.rate_ratio_lo.loc[GROUPS]
n_groups, n_higher, n_clear = len(GROUPS), int((crude.ratio.loc[GROUPS] > 1).sum()), int((lower > 1).sum())
not_clear = list(lower.index[lower <= 1])


def how_many(higher, clear, total):
    """The report's words for how many groups run higher on young machines: 'every' when every interval excludes 1,
    'nearly every' when every estimate is higher and the interval excludes 1 for all but at most two."""
    if clear == total:
        return 'every'
    if higher == total and total - clear <= 2:
        return 'nearly every'
    return f'{clear} of {total}'


groups_word = how_many(n_higher, n_clear, n_groups)
assert np.isclose(crude.at[ALL, 'ratio'], ratio_all, rtol=1e-8, atol=0), 'the age bands give another ratio for all tickets'
print(f'{n_higher} of {n_groups} request groups have a higher rate on young machines, and the 95% interval excludes '
      f"1 for {n_clear}: all but {' and '.join(not_clear)}.")
print('All tickets from the age bands give the same ratio as above.')
crude.round({'young per 100': 3, 'older per 100': 3, 'ratio': 4})

10 of 10 request groups have a higher rate on young machines, and the 95% interval excludes 1 for 8: all but Network and Software.
All tickets from the age bands give the same ratio as above.


,young tickets,older tickets,young per 100,older per 100,ratio,95% interval
type,,,,,,
Door,1193,987,2.645,1.530,1.7290,1.49–2.01
Mechanical,714,623,1.583,0.966,1.6394,1.33–2.04
Electronics,875,730,1.940,1.131,1.7146,1.43–2.10
Network,193,250,0.428,0.387,1.1043,0.83–1.45
Software,41,47,0.091,0.073,1.2478,0.68–2.28
Operator error,84,61,0.186,0.095,1.9698,1.33–2.93
Other,84,60,0.186,0.093,2.0026,1.31–3.05
No Trouble Found,265,230,0.588,0.356,1.6481,1.32–2.05
Monitoring,80,37,0.177,0.057,3.0929,1.94–5.19


## Within model: Electronics' share 1.31 times as large on young machines and Mechanical's 0.70 times

A group's share ratio within model is its share of a model's young-machine tickets ÷ its share of the model's older-machine tickets, pooled by Mantel–Haenszel over the 15 comparison models, those with 100+ installed machine-months in each age group, which we count in `ag_support.csv`. The report's Definitions do not narrow a model's tickets to some request groups, and its Methods count all unplanned tickets unless stated otherwise, so on the Definitions' base a model's tickets are all of its unplanned tickets. In its primary run, which gives the `primary` rows of `followups/ag_type_adjusted.csv` and the report's 1.31 and 0.70, `followups.py` uses a narrower base: each comparison model's unplanned tickets on its fleet machines in the nine analysis groups (Door, Mechanical, Electronics, Network, Software, Operator error, Other, No Trouble Found, and Monitoring). These are every group but Not categorized: the script sets `PRIMARY = GROUPS[:9]`, and in `part_a` it keeps those columns of the ticket counts (`Cf = Cg[:, fam]`), whose sums by model and age group are the totals n1 and n0 of the share ratio, as docs/09 §4 plans ("n1_s and n0_s are the totals over the analysis family"). `ag_support.csv` has both bases by model and age group, the nine groups as `family_tickets` and all unplanned tickets as `tickets`. The script also computes the share ratios on all ten groups, on the same machines and models, as sensitivity check S-A7 (variant `S-A7 with Not categorised`), and we confirm each base from the script's shares, which divide by the nine-group totals in its primary rows and by all tickets in its S-A7 rows.

The pooled ratio needs every comparison model's tickets by group, and the committed outputs hold these for only six models (`followups/ag_by_model.csv`). For the other nine, `followups/rp_pairs.csv` lists every ticket opened before Aug 2nd, 2026, and the next ticket on its machine, so it misses a few of their tickets. We try every way to place a group's missing tickets among the models' missing tickets and keep the placement that gives the script's within-model rate ratio, which does not depend on the base. One placement does for Electronics and one for Mechanical, so we know each comparison model's tickets in the group, young and older, and we recompute the share ratio and the excess on both bases and compare them with the script's primary and S-A7 rows. The 95% interval comes from the script's bootstrap over accounts, which it runs on the nine-group base only, and the test result comes from its permutation test.

In [5]:
import itertools

adjusted = R.read_csv(OUT, 'followups/ag_type_adjusted.csv')
primary = adjusted[adjusted.variant == 'primary']  # the comparison models: nine groups and their Categories
adj_groups = primary[primary.level == 'group'].set_index('type')
adj_cats = primary[primary.level == 'category'].set_index('type')
s_a7 = adjusted[(adjusted.variant == 'S-A7 with Not categorised') & (adjusted.level == 'group')].set_index('type')

# The comparison models: at least 100 installed machine-months in each age group
months_by = support.pivot(index='model', columns='age_group', values='months').reindex(columns=AGES).fillna(0)
comparison = sorted(months_by.index[(months_by >= definitions['min_support_months']).all(axis=1)])
assert comparison == sorted(support.model[support.in_support].unique()) == sorted(summary['facts']['support']['support_models'])
M1, M0 = months_by.loc[comparison, AGES[0]], months_by.loc[comparison, AGES[1]]  # their young and older machine-months


def base(column):
    """Each comparison model's young-machine and older-machine tickets in a column of ag_support.csv."""
    table = support[support.model.isin(comparison)].pivot(index='model', columns='age_group', values=column)
    return table[AGES[0]], table[AGES[1]]


# The two bases: the nine analysis groups (the script's primary run) and all unplanned tickets (the Definitions)
n1, n0 = base('family_tickets')
n1_all, n0_all = base('tickets')
LEFT_OUT = [g for g in GROUPS if g not in definitions['primary_family']]  # the group the nine leave out
assert len(LEFT_OUT) == 1 and len(definitions['primary_family']) == len(GROUPS) - 1
for script_rows, young_base, old_base, variant in ((adj_groups, n1, n0, 'primary'), (s_a7, n1_all, n0_all, 'S-A7')):
    assert (np.allclose(script_rows.under4_share, script_rows.under4_tickets / young_base.sum(), rtol=1e-8, atol=0)
            and np.allclose(script_rows.plus4_share, script_rows.plus4_tickets / old_base.sum(), rtol=1e-8, atol=0)), (
        f"the script's {variant} shares divide by another base")
left_out = s_a7.loc[LEFT_OUT[0]]
assert (n1_all - n1).sum() == left_out.under4_tickets and (n0_all - n0).sum() == left_out.plus4_tickets


def mh_share(a, b, n1, n0):
    """Mantel–Haenszel share ratio over the models: the sum of a·n0/N ÷ the sum of b·n1/N, where N = n1 + n0."""
    N = n1 + n0
    return (a * n0 / N).sum() / (b * n1 / N).sum()


def mh_rate(a, b):
    """Mantel–Haenszel rate ratio over the models: the sum of a·M0/M ÷ the sum of b·M1/M, where M = M1 + M0."""
    M = M1 + M0
    return (a * M0 / M).sum() / (b * M1 / M).sum()


def mh_excess(a, b, n1, n0):
    """Young machines' tickets minus those expected at the same within-model mix: the sum of a − n1·(a + b)/N."""
    return (a - n1 * (a + b) / (n1 + n0)).sum()


# The per-model counts that the committed outputs show
six = R.read_csv(OUT, 'followups/ag_by_model.csv')  # every group's tickets on the six models with 30+ in each age group
pairs = R.read_csv(OUT, 'followups/rp_pairs.csv')  # one row per ticket opened before Aug 2nd, 2026, with the next one
COLS = ['case_id', 'machine_id', 'group', 'category']
seen = pd.concat([pairs[['case_id', 'machine_id', 'index_group', 'index_category']].set_axis(COLS, axis=1),
                  pairs.loc[pairs.next_case_id.notna(), ['next_case_id', 'machine_id', 'next_group', 'next_category']]
                  .set_axis(COLS, axis=1)]).drop_duplicates('case_id')
seen = seen.merge(machines[['serial_id', 'model', 'age_group', 'install_year']], left_on='machine_id',
                  right_on='serial_id')  # fleet machines only; model, age group and install year from IPA's list
assert set(six.model) <= set(comparison)
assert (seen.groupby('machine_id').size().reindex(machines.serial_id, fill_value=0).to_numpy()
        <= machines.tickets.to_numpy()).all()  # never more tickets than the machine has
others = [m for m in comparison if m not in set(six.model)]
cells_other = pd.MultiIndex.from_product([others, AGES], names=['model', 'age_group'])
seen_other = seen[seen.model.isin(others)]
missing = listed.reindex(cells_other, fill_value=0) - seen_other.groupby(['model', 'age_group']).size().reindex(
    cells_other, fill_value=0)  # the nine models' tickets that rp_pairs.csv does not show
assert (missing >= 0).all()
room_young = missing.xs(AGES[0], level='age_group')[lambda s: s > 0]
room_old = missing.xs(AGES[1], level='age_group')[lambda s: s > 0]


def placements(total, room):
    """Every way to put `total` tickets into cells with room for the given numbers of tickets."""
    return [p for p in itertools.product(*(range(int(r) + 1) for r in room)) if sum(p) == total]


def group_counts(group):
    """The group's tickets on each comparison model, young and older: the six models' counts, the tickets that
    rp_pairs.csv shows on the other nine, and the one placement of the rest that gives the script's within-model rate
    ratio. Also returns how many young and older tickets we placed and how many placements we tried."""
    young, older = pd.Series(0, index=comparison), pd.Series(0, index=comparison)
    by_six = six.pivot(index='model', columns='age_group', values=f'tickets_{group}')
    young.loc[by_six.index], older.loc[by_six.index] = by_six[AGES[0]].to_numpy(), by_six[AGES[1]].to_numpy()
    shown = seen_other[seen_other.group == group].groupby(['model', 'age_group']).size()
    for m in others:
        young[m], older[m] = shown.get((m, AGES[0]), 0), shown.get((m, AGES[1]), 0)
    script = adj_groups.loc[group]
    left_young, left_old = int(script.under4_tickets - young.sum()), int(script.plus4_tickets - older.sum())
    tried = [(young.add(pd.Series(py, index=room_young.index), fill_value=0),
              older.add(pd.Series(po, index=room_old.index), fill_value=0))
             for py in placements(left_young, room_young) for po in placements(left_old, room_old)]
    fits = [(a, b) for a, b in tried if np.isclose(mh_rate(a, b), script.rate_ratio_mh, rtol=1e-8, atol=0)]
    assert len(fits) == 1, f"{group}: {len(fits)} of {len(tried)} placements give the script's rate ratio"
    return fits[0], left_young, left_old, len(tried)


passing = adj_groups[adj_groups.significant]
print(f"Groups that pass the within-model test: {' and '.join(passing.index)} "
      f"(Benjamini–Hochberg q = {', '.join(f'{q:.4f}' for q in passing.q_bh)}).")
print(f"Base of the script's primary run, the nine groups: {int(n1.sum()):,} tickets on young machines and "
      f'{int(n0.sum()):,} on older ones, on the {len(comparison)} comparison models. All their unplanned tickets: '
      f'{int(n1_all.sum()):,} and {int(n0_all.sum()):,}, with {int(left_out.under4_tickets)} and '
      f'{int(left_out.plus4_tickets)} more in {LEFT_OUT[0]}.')
BASES = {'nine': ("nine groups (the script's primary run)", n1, n0, adj_groups),
         'all': ("all unplanned tickets (the Definitions; the script's S-A7)", n1_all, n0_all, s_a7)}
within_model, shown_rows = {}, []
for group in ('Electronics', 'Mechanical'):
    (a, b), left_young, left_old, n_tried = group_counts(group)
    within_model[group] = {}
    for key, (label, young_base, old_base, script) in BASES.items():
        sr, ex = mh_share(a, b, young_base, old_base), mh_excess(a, b, young_base, old_base)
        assert (np.isclose(sr, script.at[group, 'share_ratio_mh'], rtol=1e-8, atol=0)
                and np.isclose(ex, script.at[group, 'excess_under4_tickets'], rtol=1e-8, atol=0)), (
            f"{group}, {label}: our counts give another share ratio or excess than the script's")
        within_model[group][key] = sr, ex
        lo, hi = script.at[group, 'share_ratio_lo'], script.at[group, 'share_ratio_hi']
        shown_rows.append({'group': group, 'base': label, 'young tickets': int(a.sum()), 'older tickets': int(b.sum()),
                           'young share': R.pct(a.sum() / young_base.sum(), 1),
                           'older share': R.pct(b.sum() / old_base.sum(), 1),
                           'share ratio within model': R.num(sr, 4), 'excess young tickets': f'{ex:+.1f}',
                           "95% interval (the script's bootstrap)": '' if np.isnan(lo) else R.span(lo, hi, 2)})
    print(f'{group}: rp_pairs.csv misses {left_young} of its young-machine and {left_old} of its older-machine tickets '
          f"on the nine other models. Of the {n_tried} ways to place them among those models' {int(room_young.sum())} "
          f"and {int(room_old.sum())} missing tickets, one gives the script's rate ratio, {mh_rate(a, b):.4f}. On those "
          f"counts the share ratio is {within_model[group]['nine'][0]:.4f} on the nine groups and "
          f"{within_model[group]['all'][0]:.4f} on all tickets, as in the script's primary and S-A7 rows.")
(sr_el, excess_el), (sr_el_all, excess_el_all) = within_model['Electronics']['nine'], within_model['Electronics']['all']
sr_mech, sr_mech_all = within_model['Mechanical']['nine'][0], within_model['Mechanical']['all'][0]
pd.DataFrame(shown_rows).set_index(['group', 'base'])

Groups that pass the within-model test: Mechanical and Electronics (Benjamini–Hochberg q = 0.0022, 0.0022).
Base of the script's primary run, the nine groups: 3,478 tickets on young machines and 2,073 on older ones, on the 15 comparison models. All their unplanned tickets: 3,574 and 2,142, with 96 and 69 more in Not categorised.


Electronics: rp_pairs.csv misses 2 of its young-machine and 5 of its older-machine tickets on the nine other models. Of the 486 ways to place them among those models' 11 and 28 missing tickets, one gives the script's rate ratio, 2.1809. On those counts the share ratio is 1.3132 on the nine groups and 1.3218 on all tickets, as in the script's primary and S-A7 rows.


Mechanical: rp_pairs.csv misses 2 of its young-machine and 6 of its older-machine tickets on the nine other models. Of the 666 ways to place them among those models' 11 and 28 missing tickets, one gives the script's rate ratio, 0.9580. On those counts the share ratio is 0.6988 on the nine groups and 0.6993 on all tickets, as in the script's primary and S-A7 rows.


young tickets  \
group       base                                                                        
Electronics nine groups (the script's primary run)                                870   
            all unplanned tickets (the Definitions; the script's S-A7)            870   
Mechanical  nine groups (the script's primary run)                                690   
            all unplanned tickets (the Definitions; the script's S-A7)            690   

                                                                        older tickets  \
group       base                                                                        
Electronics nine groups (the script's primary run)                                465   
            all unplanned tickets (the Definitions; the script's S-A7)            465   
Mechanical  nine groups (the script's primary run)                                406   
            all unplanned tickets (the Definitions; the script's S-A7)            406   

                                                                       young share  \
group       base                                                                     
Electronics nine groups (the script's primary run)                           25.0%   
            all unplanned tickets (the Definitions; the script's S-A7)       24.3%   
Mechanical  nine groups (the script's primary run)                           19.8%   
            all unplanned tickets (the Definitions; the script's S-A7)       19.3%   

                                                                       older share  \
group       base                                                                     
Electronics nine groups (the script's primary run)                           22.4%   
            all unplanned tickets (the Definitions; the script's S-A7)       21.7%   
Mechanical  nine groups (the script's primary run)                           19.6%   
            all unplanned tickets (the Definitions; the script's S-A7)       19.0%   

                                                                       share ratio within model  \
group       base                                                                                  
Electronics nine groups (the script's primary run)                                       1.3132   
            all unplanned tickets (the Definitions; the script's S-A7)                   1.3218   
Mechanical  nine groups (the script's primary run)                                       0.6988   
            all unplanned tickets (the Definitions; the script's S-A7)                   0.6993   

                                                                       excess young tickets  \
group       base                                                                              
Electronics nine groups (the script's primary run)                                    +49.3   
            all unplanned tickets (the Definitions; the script's S-A7)                +50.5   
Mechanical  nine groups (the script's primary run)                                    -61.7   
            all unplanned tickets (the Definitions; the script's S-A7)                -61.6   

                                                                       95% interval (the script's bootstrap)  
group       base                                                                                              
Electronics nine groups (the script's primary run)                                                 1.09–1.62  
            all unplanned tickets (the Definitions; the script's S-A7)                                        
Mechanical  nine groups (the script's primary run)                                                 0.59–0.84  
            all unplanned tickets (the Definitions; the script's S-A7)

**On the base the Definitions describe, Electronics' share ratio is 1.32.** The report's 1.31 is the script's primary value, 1.3132, on the nine analysis groups. With all of a model's unplanned tickets as the base, the counts above give 1.3218, as does the script's S-A7 variant, and 1.3218 rounds to 1.32. Not categorized holds 96 of the young machines' tickets and 69 of the older machines' tickets on the comparison models, and adding them to the base moves the ratio across the rounding boundary at 1.315. Mechanical's 0.70 holds on both bases (0.6988 and 0.6993), and so does "mostly Touchscreen" (see the section on the 402 tickets below). The check has rows for both of these on both bases, and this table lists Electronics' 1.31.

In [6]:
report_el = '1.31'  # the report's text
on_definitions, on_nine = R.num(sr_el_all, 2), R.num(sr_el, 2)
if on_definitions == report_el:
    reproduced = 'yes'
elif on_nine == report_el:
    reproduced = f'no: {report_el} holds only on the nine-group base, which the report does not name'
else:
    reproduced = 'no'
pd.DataFrame([{'Finding': "Within model: Electronics' share on young vs. older machines", 'Report': report_el,
               "On all of a model's unplanned tickets (the Definitions' base)": on_definitions,
               "On the nine analysis groups (the script's primary run)": on_nine,
               'Reproduced as worded': reproduced}])

,Finding,Report,On all of a model's unplanned tickets (the Definitions' base),On the nine analysis groups (the script's primary run),Reproduced as worded
0,Within model: Electronics' share on young vs. older machines,1.31,1.32,1.31,"no: 1.31 holds only on the nine-group base, which the report does not name"


## Electronics' larger share is mostly Touchscreen

"Mostly Touchscreen" refers to Electronics' excess within model: young machines' Electronics tickets minus those expected at the same within-model mix, the numerator of the script's Cochran–Mantel–Haenszel statistic. It needs every comparison model's tickets by Category, so we read each Electronics Category's excess on the nine-group base from `ag_type_adjusted.csv` (the Categories with 20+ tickets from 10+ accounts in each age group) and find the one that gives most of the group's. The script gives no Categories on all of a model's tickets, so the section on the 402 tickets below recomputes Touchscreen's excess on both bases from per-model counts.

In [7]:
el_excess = adj_cats[(adj_cats.group == 'Electronics') & adj_cats.estimable].excess_under4_tickets
top_share = el_excess.max() / excess_el
mostly = el_excess.idxmax() if top_share > 0.5 else 'no single Category'
print(f"{el_excess.idxmax()} gives {el_excess.max():+.1f} of Electronics' {excess_el:+.1f} excess young-machine tickets "
      f'on the nine groups ({R.pct(top_share)}).')
pd.DataFrame({'excess young tickets': el_excess.round(1),
              'share ratio within model': adj_cats.share_ratio_mh[el_excess.index].round(3)}).sort_values(
    'excess young tickets', ascending=False)

Touchscreen gives +44.3 of Electronics' +49.3 excess young-machine tickets on the nine groups (90%).


,excess young tickets,share ratio within model
type,,
Touchscreen,44.3,1.853
Power Supply,6.7,1.581
Badge Reader,-3.9,0.772
Computer Board,-6.0,0.911


## 80% of the 1,447 extra tickets come from young machines running above older ones of the same model

Extra tickets are young machines' tickets minus their machine-months times the older machines' rate, and they split exactly into a within-model part (on the comparison models, young machines' tickets minus their machine-months times the same model's older-machine rate), a model-mix part, and a part for the other models. `ag_support.csv` has the tickets and machine-months by model and age group that the split needs, and we compare the parts with `followups/ag_decomposition.csv` and `followups_summary.json`.

In [8]:
tickets_by = support.pivot(index='model', columns='age_group', values='tickets').reindex(columns=AGES).fillna(0)
inside = tickets_by.index.isin(comparison)
old_rate = tickets_by[AGES[1]].sum() / months_by[AGES[1]].sum()  # older machines' rate, all models
own_rate = tickets_by.loc[inside, AGES[1]] / months_by.loc[inside, AGES[1]]  # each comparison model's older-machine rate
expected_same_model = (months_by.loc[inside, AGES[0]] * own_rate).sum()

extra = tickets_by[AGES[0]].sum() - months_by[AGES[0]].sum() * old_rate
within = tickets_by.loc[inside, AGES[0]].sum() - expected_same_model
mix = expected_same_model - months_by.loc[inside, AGES[0]].sum() * old_rate
outside = tickets_by.loc[~inside, AGES[0]].sum() - months_by.loc[~inside, AGES[0]].sum() * old_rate
share_within = within / extra

dec = R.read_csv(OUT, 'followups/ag_decomposition.csv')
dec_all = dec[dec.level == 'all'].iloc[0]
part_a = summary['facts']['part_a']
ours = [extra, within, mix, outside]
assert np.isclose(within + mix + outside, extra), 'the parts do not add up to the extra tickets'
assert np.allclose(ours, dec_all[['extra_tickets', 'within_model_age_tickets', 'model_mix_tickets',
                                  'outside_support_tickets']].astype(float)), 'the parts differ from ag_decomposition.csv'
assert np.allclose(ours, [part_a[k] for k in ('extra_tickets', 'extra_within_model_age', 'extra_model_mix',
                                              'extra_outside_support')]), 'the parts differ from followups_summary.json'
print('The parts add up to the extra tickets, and they match ag_decomposition.csv and followups_summary.json.')
pd.DataFrame({'tickets': [R.num(x, 1) for x in ours], 'share of the extra': [R.pct(x / extra, 1) for x in ours]},
             index=['extra tickets on young machines', 'within model: young above older machines of the same model',
                    'model mix: young machines more often on high-rate models',
                    f'models outside the {len(comparison)} comparison models'])

The parts add up to the extra tickets, and they match ag_decomposition.csv and followups_summary.json.


,tickets,share of the extra
extra tickets on young machines,"1,447.2",100.0%
within model: young above older machines of the same model,"1,156.1",79.9%
model mix: young machines more often on high-rate models,282.2,19.5%
models outside the 15 comparison models,8.9,0.6%


## Touchscreen tickets cluster on 2022–23 installs: 1.54 per 100, 3.9 times the 2024–26 rate

`followups/ag_install_year.csv` counts Touchscreen tickets and machine-months by comparison model and install year, at any age, so we add up the years into the report's four install cohorts (the cohort labels of `followups/ag_install_pooled.csv`) and divide. The cohorts were chosen after seeing the data, so the result is descriptive, and we compare our sums with the pooled table. The cohorts are fixed in `followups.py` (`INSTALL_COHORTS`), so the check has no row for the newest cohort, and its row for the ratio names that cohort as the pooled table labels it.

In [9]:
years = R.read_csv(OUT, 'followups/ag_install_year.csv')  # three Categories by comparison model and install year
ts_years = years[years.category == 'Touchscreen']
pooled = R.read_csv(OUT, 'followups/ag_install_pooled.csv')
pooled_ts = pooled[(pooled.category == 'Touchscreen') & (pooled.unit == 'cohort')].set_index('install')
COHORTS = list(pooled_ts.index)


def cohort_years(label):
    """'2022-23' -> (2022, 2023)."""
    first, last = label.split('-')
    return int(first), int(first[:2] + last)


def dashed(label):
    """'2022-23' -> '2022–23', as the report prints a span of years."""
    return label.replace('-', '–')


cohorts = pd.DataFrame([ts_years[ts_years.install_year.between(*cohort_years(c))][
    ['machines', 'under4_machines', 'months', 'tickets']].sum().rename(c) for c in COHORTS])
cohorts = cohorts.astype({'machines': int, 'under4_machines': int, 'tickets': int})
cohorts['per 100 machine-months'] = 100 * cohorts.tickets / cohorts.months
newest = COHORTS[-1]
cohorts[f'vs. {dashed(newest)}'] = cohorts['per 100 machine-months'] / cohorts.at[newest, 'per 100 machine-months']
cohorts['share of machines under 4'] = cohorts.under4_machines / cohorts.machines
cluster = cohorts['per 100 machine-months'].idxmax()
rate_cluster = cohorts.at[cluster, 'per 100 machine-months']
ratio_cluster = cohorts.at[cluster, f'vs. {dashed(newest)}']

same = (np.allclose(cohorts[['machines', 'months', 'tickets', 'per 100 machine-months']].to_numpy(float),
                    pooled_ts.loc[COHORTS, ['machines', 'months', 'tickets', 'rate_per_100']].to_numpy(float))
        and np.allclose(cohorts[f'vs. {dashed(newest)}'], pooled_ts.loc[COHORTS, 'ratio_to_newest_cohort']))
assert same, 'our cohort sums differ from ag_install_pooled.csv'
print(f'Highest Touchscreen rate: {dashed(cluster)} installs, {rate_cluster:.3f} per 100 machine-months, '
      f'{ratio_cluster:.3f} times the {dashed(newest)} rate. Our cohort sums match ag_install_pooled.csv.')
cohorts.drop(columns='under4_machines').rename(index=dashed).round(3)

Highest Touchscreen rate: 2022–23 installs, 1.538 per 100 machine-months, 3.895 times the 2024–26 rate. Our cohort sums match ag_install_pooled.csv.


,machines,months,tickets,per 100 machine-months,vs. 2024–26,share of machines under 4
2016–18,1003,13049.298,27,0.207,0.524,0.000
2019–21,1971,25643.236,68,0.265,0.671,0.000
2022–23,1599,20803.417,320,1.538,3.895,0.954
2024–26,2377,24309.487,96,0.395,1.000,1.000


## Electronics runs 1.71 times as often on young machines, near the 1.66 for all tickets

These are the crude rate ratios per installed machine-month of the Electronics group and of all unplanned tickets, all models together, from the band sums of `ag_band_rates.csv` above. Electronics' share of all unplanned tickets, all models together, is nearly the same at both ages, which is why its ratio is close to that of all tickets.

In [10]:
ratio_el = crude.at['Electronics', 'ratio']
share_el_young = crude.at['Electronics', 'young tickets'] / crude.at[ALL, 'young tickets']
share_el_old = crude.at['Electronics', 'older tickets'] / crude.at[ALL, 'older tickets']
print(f'Electronics: {ratio_el:.4f} times per machine-month vs. {ratio_all:.4f} for all tickets. Electronics is '
      f"{R.pct(share_el_young, 1)} of young machines' unplanned tickets vs. {R.pct(share_el_old, 1)} of older "
      "machines', all models and all ten request groups together.")

Electronics: 1.7146 times per machine-month vs. 1.6633 for all tickets. Electronics is 24.1% of young machines' unplanned tickets vs. 23.4% of older machines', all models and all ten request groups together.


## Touchscreen stands out at 4.2 times, with 411 tickets on young machines

`followups/ag_type_rates.csv` counts each Electronics Category's tickets on young and older machines, all models together, and we divide them by the machine-months above. Touchscreen stands out when its ratio is the highest of the Electronics Categories with at least 30 tickets in each age group (the script's threshold for showing a model or type).

In [11]:
el_rows = rates[(rates.level == 'category') & (rates.group == 'Electronics')].set_index('type')
el_cat = pd.DataFrame({'young tickets': el_rows.under4_tickets, 'older tickets': el_rows.plus4_tickets})
el_cat['young per 100'] = 100 * el_cat['young tickets'] / months_young
el_cat['older per 100'] = 100 * el_cat['older tickets'] / months_old
el_cat['ratio'] = el_cat['young per 100'] / el_cat['older per 100']

min_tickets = definitions['min_shown_tickets']
shown_cats = el_cat[(el_cat['young tickets'] >= min_tickets) & (el_cat['older tickets'] >= min_tickets)]
stands_out = shown_cats.ratio.idxmax()
rest = shown_cats.drop(stands_out)
ratio_ts = el_cat.at['Touchscreen', 'ratio']
ts_young = int(el_cat.at['Touchscreen', 'young tickets'])
print(f'{stands_out}: {shown_cats.ratio.max():.3f} times. The other Electronics Categories with {min_tickets}+ tickets '
      f'in each age group: {R.span(rest.ratio.min(), rest.ratio.max(), 2)} times.')
shown_cats.round(3)

Touchscreen: 4.230 times. The other Electronics Categories with 30+ tickets in each age group: 0.97–1.75 times.


,young tickets,older tickets,young per 100,older per 100,ratio
type,,,,,
Computer Board,259,383,0.574,0.594,0.967
Touchscreen,411,139,0.911,0.215,4.230
Power Supply,83,106,0.184,0.164,1.120
Badge Reader,60,49,0.133,0.076,1.752


## 344 connector reseats or replacements

The 344 are the young-machine Touchscreen tickets with Sub Category Connector Reseat/Replace. Part A's tables have no Sub Category, so the case-list builder counts them from IPA's case export and writes them, with the other subsets of its Summary (table 7), to `electronics/ec_subsets.csv`. Its count of young-machine Touchscreen tickets equals the 411 above.

In [12]:
subsets = R.read_csv(OUT, 'electronics/ec_subsets.csv').set_index('key')
assert subsets.at['u4_focus', 'tickets'] == ts_young
n_conn = int(subsets.at['u4_conn', 'tickets'])
print(f"{n_conn} of the {ts_young} young-machine Touchscreen tickets are "
      f"{subsets.at['u4_conn', 'label'].removeprefix('of which ')}.")
subsets.loc[['u4', 'u4_focus', 'u4_conn', 'u4_coh'], ['label', 'tickets', 'machines', 'accounts']]

344 of the 411 young-machine Touchscreen tickets are Connector Reseat/Replace.


,label,tickets,machines,accounts
key,,,,
u4,"Electronics, under 4",875,640,316
u4_focus,"Touchscreen, under 4",411,313,184
u4_conn,of which Connector Reseat/Replace,344,268,163
u4_coh,of which installed 2022-23,315,232,125


## 402 on the four 2.0 models and 315 on 2022–23 installs

`ag_install_year.csv` counts Touchscreen tickets by comparison model and install year, and every install year but 2022 has only young or only older machines, so only the 2022 counts need splitting by age: we count the older machines' Touchscreen tickets on 2022 installs in `rp_pairs.csv` (each machine's age and install year from `fx_machine_flags.csv`) and check that this count is complete. We also check that all 411 young-machine Touchscreen tickets are on the comparison models and that the per-model counts give the script's within-model share ratio and excess for Touchscreen on the nine-group base. On all of a model's tickets, the base the Definitions describe, we compute Touchscreen's excess from the same counts and divide it by Electronics' excess on that base from the within-model section. The four 2.0 models are the scrubEx family's models whose names end in 2.0 (`fleet/fx_model_rates.csv`).

In [13]:
# The one install year whose machines are partly young and partly older
mixed = ts_years.install_year[(ts_years.under4_machines > 0) & (ts_years.under4_machines < ts_years.machines)].unique()
assert len(mixed) == 1
MIXED = int(mixed[0])
before, after = ts_years.install_year < MIXED, ts_years.install_year > MIXED
assert (ts_years.under4_machines[before] == 0).all() and (ts_years.under4_machines[after] == ts_years.machines[after]).all()

# Older machines' Touchscreen tickets on that year's installs, ticket by ticket, and a check that none is missing
ts_seen = seen[(seen.category == 'Touchscreen') & seen.model.isin(comparison)]
old_mixed = ts_seen[(ts_seen.install_year == MIXED) & (ts_seen.age_group == AGES[1])].groupby('model').size()
assert int(old_mixed.sum()) == adj_cats.at['Touchscreen', 'plus4_tickets'] - ts_years.tickets[before].sum(), (
    f"rp_pairs.csv misses some of older machines' Touchscreen tickets on {MIXED} installs")

assert sorted(ts_years.model.unique()) == comparison
old_mixed = old_mixed.reindex(comparison, fill_value=0)


def tickets_per_model(rows):
    return ts_years[rows].groupby('model').tickets.sum().reindex(comparison, fill_value=0)


first, last = cohort_years(cluster)
assert first <= MIXED <= last and last + 1 == cohort_years(newest)[0]
assert cohort_years(newest)[1] >= ts_years.install_year.max()
by_model = pd.DataFrame({
    'on older machines': tickets_per_model(before) + old_mixed,
    f'young, {dashed(cluster)} installs': tickets_per_model(ts_years.install_year.between(first, last)) - old_mixed,
    f'young, {dashed(newest)} installs': tickets_per_model(ts_years.install_year.between(*cohort_years(newest))),
    'on young machines': tickets_per_model(ts_years.install_year >= MIXED) - old_mixed})
assert (by_model['on young machines'] == by_model.iloc[:, 1] + by_model.iloc[:, 2]).all()

# The 2.0 models: the scrubEx family's models whose names end in 2.0
model_rates = R.read_csv(OUT, 'fleet/fx_model_rates.csv')  # one row per model, with its family
v2 = sorted(model_rates.model[(model_rates.family == 'scrubEx') & model_rates.model.str.endswith('2.0')])
v2_order = by_model.loc[v2, 'on young machines'].sort_values(ascending=False).index.tolist()
n_v2 = int(by_model.loc[v2, 'on young machines'].sum())
n_cluster = int(by_model[f'young, {dashed(cluster)} installs'].sum())
n_young_ts = int(by_model['on young machines'].sum())
assert n_young_ts == ts_young, 'some young-machine Touchscreen tickets are off the comparison models'

# The per-model counts give the script's within-model share ratio and excess for Touchscreen (nine groups)
a, b = by_model['on young machines'], by_model['on older machines']
sr_ts, excess_ts = mh_share(a, b, n1, n0), mh_excess(a, b, n1, n0)
assert (np.isclose(sr_ts, adj_cats.at['Touchscreen', 'share_ratio_mh'], rtol=1e-8, atol=0)
        and np.isclose(excess_ts, adj_cats.at['Touchscreen', 'excess_under4_tickets'], rtol=1e-8, atol=0)), (
    "the per-model counts give another share ratio or excess for Touchscreen than the script's")
# On all of a model's tickets, where the script gives no Categories: Touchscreen's part of Electronics' excess
excess_ts_all = mh_excess(a, b, n1_all, n0_all)
share_ts_all = excess_ts_all / excess_el_all
mostly_all = 'Touchscreen' if share_ts_all > 0.5 else 'no single Category'
print(f"{MIXED} is the only install year that mixes ages, and rp_pairs.csv has all {int(old_mixed.sum())} of the older "
      f"machines' Touchscreen tickets on {MIXED} installs.")
print(f'All {ts_young} young-machine Touchscreen tickets (ag_type_rates.csv) are on the comparison models.')
builder = R.read_csv(OUT, 'electronics/ec_counts.csv').set_index('key').value
assert builder['focus_under4_core_models'] == n_v2 and subsets.at['u4_coh', 'tickets'] == n_cluster
print(f"The case-list builder's counts from IPA's case export agree: {n_v2} on the 2.0 models and {n_cluster} on "
      f'{dashed(cluster)} installs.')
print(f'On the nine groups, the within-model share ratio {sr_ts:.4f} and excess {excess_ts:+.2f} match '
      f"ag_type_adjusted.csv. On all of a model's tickets, Touchscreen gives {excess_ts_all:+.1f} of Electronics' "
      f'{excess_el_all:+.1f} excess young-machine tickets ({R.pct(share_ts_all)}).')

WORDS = {1: 'one', 2: 'two', 3: 'three', 4: 'four', 5: 'five', 6: 'six', 7: 'seven', 8: 'eight', 9: 'nine'}


def and_join(names):
    """['a', 'b', 'c'] -> 'a, b, and c'."""
    return ' and '.join(names) if len(names) < 3 else ', '.join(names[:-1]) + f', and {names[-1]}'


table = by_model[by_model['on young machines'] > 0].sort_values('on young machines', ascending=False)
table.insert(0, 'scrubEx 2.0 model', np.where(table.index.isin(v2), 'yes', 'no'))
table.loc[f'All {len(comparison)} comparison models'] = ['', *by_model.sum()]
table

2022 is the only install year that mixes ages, and rp_pairs.csv has all 5 of the older machines' Touchscreen tickets on 2022 installs.
All 411 young-machine Touchscreen tickets (ag_type_rates.csv) are on the comparison models.
The case-list builder's counts from IPA's case export agree: 402 on the 2.0 models and 315 on 2022–23 installs.
On the nine groups, the within-model share ratio 1.8530 and excess +44.32 match ag_type_adjusted.csv. On all of a model's tickets, Touchscreen gives +44.9 of Electronics' +50.5 excess young-machine tickets (89%).


,scrubEx 2.0 model,on older machines,"young, 2022–23 installs","young, 2024–26 installs",on young machines
model,,,,,
LVD-Remote-2.0,yes,32,141,37,178
MV-2.0,yes,11,109,29,138
LVRR-WP-2.0,yes,18,53,21,74
SV-48-2.0,yes,1,8,4,12
alEx LC-D,no,1,1,2,3
alEx MC-S,no,0,1,1,2
alEx MC-D,no,0,0,2,2
A-LC,no,2,1,0,1
alEx SD 2.0,no,1,1,0,1


## The step from 0.36 to 1.61 per 100 between Jan–Apr and May–Aug 2022 installs

The step compares the comparison models' machines installed January to April 2022 with those installed May to August 2022, by their Touchscreen tickets at any age per 100 installed machine-months. Part A's tables give install dates only as years, so the case-list builder counts each window from IPA's installed-base list and writes the counts to `electronics/ec_edge.csv`, and we recompute the rates from them. The builder checks that its counts by install month add up, for 2022, to `followups/ag_install_pooled.csv`, and here we check that the two windows fit inside that 2022 row. The windows' install dates are four months apart, so their machines are nearly the same age. The 4-year line falls inside the first window, which is why only some of its machines count as young.

The two windows are fixed in the builder (`EDGE` in `build_electronics_cases.py`) and were chosen after seeing the data (docs/10 §6), so the step is descriptive, as the report's Definitions say. The check has no row for the windows themselves, and its rows for the two rates name the windows as `ec_edge.csv` labels them.

In [14]:
edge = R.read_csv(OUT, 'electronics/ec_edge.csv')
edge['per 100'] = 100 * edge.tickets / edge.months
assert np.allclose(edge['per 100'], edge.rate_per_100)
rate_early, rate_late = edge['per 100']
year = R.read_csv(OUT, 'followups/ag_install_pooled.csv', dtype={'install': str}).query(
    "category == 'Touchscreen' and unit == 'year' and install == '2022'").iloc[0]
assert edge.machines.sum() <= year.machines and edge.months.sum() <= year.months and edge.tickets.sum() <= year.tickets


def short(label):
    """'January to April 2022' -> ('Jan–Apr', '2022')."""
    first, last = label.split(' to ')
    return f'{first[:3]}–{last[:3]}', last.split()[-1]


(early, y0), (late, y1) = map(short, edge.label)
print(f'Touchscreen tickets per 100 machine-months: {rate_early:.2f} on {edge.label[0]} installs and '
      f'{rate_late:.2f} on {edge.label[1]} installs ({rate_late / rate_early:.1f} times).')
edge[['label', 'machines', 'under4', 'months', 'tickets', 'per 100']].round(2)

Touchscreen tickets per 100 machine-months: 0.36 on January to April 2022 installs and 1.61 on May to August 2022 installs (4.4 times).


,label,machines,under4,months,tickets,per 100
0,January to April 2022,232,158,3018.38,11,0.36
1,May to August 2022,253,253,3291.60,53,1.61


## All 1,605 unplanned Electronics cases on the fleet machines behind slide 4

The case list holds every unplanned ticket in the Electronics request group on the fleet machines of Part A (slide 4's population), one row per case. It is not committed, so we count those tickets over the seven age bands of `ag_band_rates.csv` and compare the count with `followups/ag_by_status.csv`, which splits the same tickets by service status.

In [15]:
el_bands = bands[bands.type == 'Electronics']
n_el = int(el_bands.tickets.sum())
n_el_young = int(el_bands.tickets[el_bands.age_group == AGES[0]].sum())
status = R.read_csv(OUT, 'followups/ag_by_status.csv')  # the same tickets by service status and age group
assert n_el == int(status.tickets[status.type == 'Electronics'].sum()), 'ag_by_status.csv gives another total'
print(f'{n_el:,} unplanned Electronics tickets on the {int(el_bands.machines.sum()):,} fleet machines: {n_el_young:,} on '
      f'young machines and {n_el - n_el_young:,} on older ones. ag_by_status.csv gives the same total.')
assert builder['tickets'] == n_el and builder['under4'] == n_el_young
print(f"The case-list builder lists the same {n_el:,} cases ({builder['under4']:,} on young machines).")

1,605 unplanned Electronics tickets on the 8,946 fleet machines: 875 on young machines and 730 on older ones. ag_by_status.csv gives the same total.
The case-list builder lists the same 1,605 cases (875 on young machines).


## Check against the report

The check leaves out Electronics' within-model share ratio (1.31), which holds only on the nine-group base and which we list in its own table under the within-model results. It has rows for "mostly Touchscreen" and for Mechanical's share ratio on both bases. The install cohorts and the two install windows are fixed in `followups.py` and in the case-list builder, so the check has no rows for them, and the rows for the rates name them as the tables label them.

In [16]:
R.check([
    ('Young machines: unplanned tickets per 100 machine-months', '8.0', R.num(rate_young, 1)),
    ('Older machines: unplanned tickets per 100 machine-months', '4.8', R.num(rate_old, 1)),
    ('Young vs. older rate, all tickets', '1.66', R.num(ratio_all, 2)),
    ('Request groups with a higher rate on young machines', 'nearly every', groups_word),
    ("Category behind most of Electronics' larger share, on all of a model's tickets (Definitions)", 'Touchscreen',
     mostly_all),
    ("Category behind most of Electronics' larger share, on the nine groups (the script's primary run)", 'Touchscreen',
     mostly),
    ("Within model: Mechanical's share on young vs. older machines, on all of a model's tickets (Definitions)", '0.70',
     R.num(sr_mech_all, 2)),
    ("Within model: Mechanical's share on young vs. older machines, on the nine groups (the script's primary run)",
     '0.70', R.num(sr_mech, 2)),
    ('Share of the extra tickets from young machines above older ones of the same model', '80%', R.pct(share_within)),
    ('Extra tickets on young machines', '1,447', R.num(extra)),
    ('Install cohort where Touchscreen tickets cluster', '2022–23', dashed(cluster)),
    ('Touchscreen tickets per 100 machine-months on those installs', '1.54', R.num(rate_cluster, 2)),
    (f"That rate vs. the newest cohort's ({dashed(newest)} installs)", '3.9', R.num(ratio_cluster, 1)),
    ('Electronics: young vs. older rate per machine-month', '1.71', R.num(ratio_el, 2)),
    ('All tickets: young vs. older rate (second paragraph)', '1.66', R.num(ratio_all, 2)),
    ('Touchscreen: young vs. older rate per machine-month', '4.2', R.num(ratio_ts, 1)),
    ('Electronics Category that stands out (highest ratio, 30+ tickets in each age group)', 'Touchscreen', stands_out),
    ('Touchscreen tickets on young machines', '411', R.num(ts_young)),
    ('Young-machine Touchscreen tickets that are connector reseats or replacements', '344', R.num(n_conn)),
    ('Young-machine Touchscreen tickets on the 2.0 models', '402', R.num(n_v2)),
    ('Number of 2.0 models (scrubEx models named 2.0)', 'four', WORDS[len(v2)]),
    ('The 2.0 models, most young-machine Touchscreen tickets first', 'LVD-Remote-2.0, MV-2.0, LVRR-WP-2.0, and SV-48-2.0',
     and_join(v2_order)),
    ('Young-machine Touchscreen tickets on 2022–23 installs', '315', R.num(n_cluster)),
    (f'Touchscreen per 100 machine-months on {early} {y0} installs', '0.36', R.num(rate_early, 2)),
    (f'Touchscreen per 100 machine-months on {late} {y1} installs', '1.61', R.num(rate_late, 2)),
    ('Unplanned Electronics cases on the fleet machines behind slide 4', '1,605', R.num(n_el)),
    ('Comparison models for within-model results (Definitions, Within model)', '15', R.num(len(comparison))),
    ('Extra tickets within model (Definitions, Extra tickets and the 80%)', '1,156', R.num(within)),
]);

,Finding,Report,Recomputed,Match
0,Young machines: unplanned tickets per 100 machine-months,8.0,8.0,yes
1,Older machines: unplanned tickets per 100 machine-months,4.8,4.8,yes
2,"Young vs. older rate, all tickets",1.66,1.66,yes
3,Request groups with a higher rate on young machines,nearly every,nearly every,yes
4,"Category behind most of Electronics' larger share, on all of a model's tickets (Definitions)",Touchscreen,Touchscreen,yes
5,"Category behind most of Electronics' larger share, on the nine groups (the script's primary run)",Touchscreen,Touchscreen,yes
6,"Within model: Mechanical's share on young vs. older machines, on all of a model's tickets (Definitions)",0.70,0.70,yes
7,"Within model: Mechanical's share on young vs. older machines, on the nine groups (the script's primary run)",0.70,0.70,yes
8,Share of the extra tickets from young machines above older ones of the same model,80%,80%,yes
9,Extra tickets on young machines,"1,447","1,447",yes


All 28 values match the report.
